In [1]:
import pandas as pd
import numpy as np

# ============================================================
# 공통 설정
# ============================================================
DATA_DIR = "./data"

# 트레이드 필터: (텍사스 합류일, 텍사스 방출일)  None = 제한 없음
TRADE_FILTER_BATTERS = {
    'Dylan Moore':      ('2025-08-21', None),
    'Donovan Solano':   ('2025-08-25', None),
    'Rowdy Tellez':     ('2025-06-18', None),
    'Leody Taveras':    (None, '2025-05-04'),
    'Blaine Crim':      (None, '2025-09-11'),
    'Jonathan Ornelas': (None, '2025-08-06'),
}
TRADE_FILTER_PITCHERS = {
    'Merrill Kelly':  ('2025-08-01', None),
    'Danny Coulombe': ('2025-08-01', None),
    'Phil Maton':     ('2025-08-01', None),
}

# 집계 파일 목록 및 불필요 컬럼
AGG_FILES = {
    'rangers_hitting_statcast.csv':    {'drop_col': 'Unnamed: 26_level_1'},
    'rangers_hitting_batted.csv':      {},
    'rangers_hitting_discipline.csv':  {},
    'rangers_pitching_statcast.csv':   {'drop_col': 'Unnamed: 26_level_1'},
    'rangers_pitching_batted.csv':     {},
    'rangers_pitching_discipline.csv': {},
}

# 반올림 설정
BATTER_ROUND = {3: ['EV','Max_EV','LA','xBA','xSLG','xwOBA'],
                2: ['Bat_Speed','Swing_Length','Attack_Angle'],
                1: ['Barrel_pct','Hard_Hit_pct','Sweet_Spot_pct','Whiff_pct']}
PITCHER_ROUND = {3: ['Avg_Velo','Max_Velo','EV_allowed','xBA_against','xwOBA_against'],
                 2: ['Avg_Spin','V_Break','H_Break','Release_Extension','IP_float'],
                 1: ['Whiff_pct','Hard_Hit_allowed_pct']}


In [2]:
# ============================================================
# 이름 정제 + 집계 파일 클리닝
# ============================================================

def fix_name(name):
    """'Lastname, Firstname' → 'Firstname Lastname'"""
    name = str(name).strip().strip('"')
    if ',' in name:
        last, first = name.split(',', 1)
        return first.strip() + ' ' + last.strip()
    return name


def fix_encoding(df, col, replacements=None):
    """
    인코딩 깨진 문자열 일괄 교정
    replacements: [(old, new), ...]  기본값: García 깨짐 교정
    """
    if replacements is None:
        replacements = [('GarcÃ­a', 'García')]
    for old, new in replacements:
        df[col] = df[col].str.replace(old, new, regex=False)
    return df


def clean_agg_files(data_dir, files_config):
    """
    집계 파일 일괄 처리:
    - Player 이름 형식 통일 (fix_name)
    - Rangers / MLB 요약 행 제거
    - 불필요 컬럼 제거
    """
    SUMMARY_ROWS = {'Rangers', 'MLB'}
    for fname, opts in files_config.items():
        path = f"{data_dir}/{fname}"
        df   = pd.read_csv(path, encoding='utf-8')
        df['Player'] = df['Player'].apply(fix_name)
        df = df[~df['Player'].isin(SUMMARY_ROWS)].reset_index(drop=True)
        drop_col = opts.get('drop_col')
        if drop_col and drop_col in df.columns:
            df = df.drop(columns=[drop_col])
        df.to_csv(path, index=False, encoding='utf-8')
        print(f"  ✓ {fname:<45} {df.shape[0]}행 {df.shape[1]}열")


In [3]:
# ============================================================
# 게임로그 전처리
# ============================================================

def ip_to_float(ip_val):
    """'1.2' → 1 + 2/3 = 1.667  (야구 이닝 표기 → 실수)"""
    try:
        s = str(ip_val)
        inn, outs = s.split('.') if '.' in s else (s, '0')
        return int(inn) + int(outs) / 3
    except:
        return np.nan


def add_pergame_stats_pitcher(df):
    """투수 게임로그: 누적 ERA/WHIP 제거 후 경기별 ERA/WHIP 추가"""
    df = df.drop(columns=[c for c in ['ERA', 'WHIP'] if c in df.columns])
    ip = df['IP'].apply(ip_to_float)
    df['game_ERA']  = np.where(ip > 0, ((df['ER'] / ip) * 9).round(2),  np.nan)
    df['game_WHIP'] = np.where(ip > 0, ((df['H'] + df['BB']) / ip).round(2), np.nan)
    return df


def add_pergame_stats_batter(df):
    """
    타자 게임로그: 누적 AVG/OBP/SLG/OPS 제거 후 경기별 값 재계산 
    """
    df = df.drop(columns=[c for c in ['AVG', 'OBP', 'SLG', 'OPS',
                                       'game_AVG', 'game_OBP', 'game_SLG', 'game_OPS']
                           if c in df.columns])

    singles = df['H'] - df['2B'] - df['3B'] - df['HR']

    # game_AVG = H / AB
    df['game_AVG'] = np.where(df['AB'] > 0,
                               (df['H'] / df['AB']).round(3), np.nan)

    # game_OBP = (H + BB + HBP) / (AB + BB + HBP)
    obp_denom = df['AB'] + df['BB'] + df['HBP']
    df['game_OBP'] = np.where(obp_denom > 0,
                               ((df['H'] + df['BB'] + df['HBP']) / obp_denom).round(3),
                               np.nan)

    # game_SLG = 총루타 / AB
    tb = singles + 2*df['2B'] + 3*df['3B'] + 4*df['HR']
    df['game_SLG'] = np.where(df['AB'] > 0,
                               (tb / df['AB']).round(3), np.nan)

    # game_OPS = game_OBP + game_SLG
    df['game_OPS'] = (df['game_OBP'].fillna(0) + df['game_SLG'].fillna(0)).round(3)
    df['game_OPS'] = df['game_OPS'].where(
        df['game_OBP'].notna() | df['game_SLG'].notna(), np.nan)

    return df


# 하위 호환용 alias
def add_pergame_stats(df):
    return add_pergame_stats_pitcher(df)


def apply_trade_filter(df, trade_filter, name_col='name', date_col='Date'):
    """
    트레이드 기준 날짜로 해당 선수의 타팀 경기 행 제거
    trade_filter: {name: (join_date | None, leave_date | None)}
    """
    df = df.sort_values(by=date_col).copy()
    mask = pd.Series(True, index=df.index)
    for player, (start, end) in trade_filter.items():
        if start:
            mask &= ~((df[name_col] == player) & (df[date_col] < start))
        if end:
            mask &= ~((df[name_col] == player) & (df[date_col] > end))
    return df[mask].reset_index(drop=True)


def round_columns(df, round_config):
    """round_config: {소수점자리: [컬럼명, ...]}"""
    for digits, cols in round_config.items():
        for col in cols:
            if col in df.columns:
                df[col] = df[col].round(digits)
    return df


def drop_unnamed(df):
    """Unnamed 컬럼 일괄 제거"""
    unnamed = [c for c in df.columns if 'Unnamed' in str(c)]
    return df.drop(columns=unnamed) if unnamed else df

In [4]:
# ============================================================
# 전체 전처리 파이프라인
# ============================================================

def preprocess_all(data_dir=DATA_DIR):

    print("─── 1. 집계 파일 이름 정제 ──────────────────")
    clean_agg_files(data_dir, AGG_FILES)

    print("\n─── 2. 투수 게임로그 전처리 ─────────────────")
    pit_path = f"{data_dir}/rangers_pitcher_gamelogs.csv"
    pit_gl   = pd.read_csv(pit_path)
    pit_gl   = add_pergame_stats_pitcher(pit_gl)
    pit_gl   = apply_trade_filter(pit_gl, TRADE_FILTER_PITCHERS)
    pit_gl   = drop_unnamed(pit_gl)
    pit_gl.to_csv(pit_path, index=False, encoding='utf-8')
    print(f"  투수 게임로그: {pit_gl.shape}")

    print("\n─── 3. 타자 게임로그 전처리 ─────────────────")
    bat_path = f"{data_dir}/rangers_batter_gamelogs.csv"
    bat_gl   = pd.read_csv(bat_path)
    bat_gl   = add_pergame_stats_batter(bat_gl)   # game_AVG/OBP/SLG/OPS 재계산
    bat_gl   = apply_trade_filter(bat_gl, TRADE_FILTER_BATTERS)
    bat_gl   = drop_unnamed(bat_gl)
    bat_gl.to_csv(bat_path, index=False, encoding='utf-8')
    print(f"  타자 게임로그: {bat_gl.shape}")

    print("\n─── 4. 일별 Statcast 반올림 ─────────────────")
    for fname, round_cfg in [
        (f"{data_dir}/rangers_2025_batters_daily_final.csv",  BATTER_ROUND),
        (f"{data_dir}/rangers_2025_pitchers_daily_final.csv", PITCHER_ROUND),
    ]:
        df = pd.read_csv(fname, encoding='utf-8')
        df = round_columns(df, round_cfg)
        df.to_csv(fname, index=False, encoding='utf-8')
        print(f"  ✓ {fname.split('/')[-1]:<45} {df.shape[0]}행")

    print("\n─── 5. 인코딩 수정 (García) ─────────────────")
    encoding_targets = [
        (f"{data_dir}/rangers_batter_gamelogs.csv",  'name'),
        (f"{data_dir}/rangers_pitcher_gamelogs.csv", 'name'),
        (f"{data_dir}/rangers_roster_2025.csv",      'name'),
        (f"{data_dir}/rangers_running.csv",           'Player'),
    ]
    for path, col in encoding_targets:
        df = pd.read_csv(path)
        df = fix_encoding(df, col)
        df.to_csv(path, index=False, encoding='utf-8')
        print(f"  ✓ {path.split('/')[-1]}")

    print("\n✅  전처리 완료")


# ── 실행 ──────────────────────────────────────────────────────
preprocess_all()
#
# 개별 실행 예시:
# bat_gl = add_pergame_stats_batter(pd.read_csv(f"{DATA_DIR}/rangers_batter_gamelogs.csv"))
# pit_gl = add_pergame_stats_pitcher(pd.read_csv(f"{DATA_DIR}/rangers_pitcher_gamelogs.csv"))

─── 1. 집계 파일 이름 정제 ──────────────────
  ✓ rangers_hitting_statcast.csv                  22행 26열
  ✓ rangers_hitting_batted.csv                    22행 17열
  ✓ rangers_hitting_discipline.csv                22행 14열
  ✓ rangers_pitching_statcast.csv                 21행 26열
  ✓ rangers_pitching_batted.csv                   21행 17열
  ✓ rangers_pitching_discipline.csv               21행 14열

─── 2. 투수 게임로그 전처리 ─────────────────
  투수 게임로그: (703, 20)

─── 3. 타자 게임로그 전처리 ─────────────────
  타자 게임로그: (1715, 23)

─── 4. 일별 Statcast 반올림 ─────────────────
  ✓ rangers_2025_batters_daily_final.csv          1665행
  ✓ rangers_2025_pitchers_daily_final.csv         685행

─── 5. 인코딩 수정 (García) ─────────────────
  ✓ rangers_batter_gamelogs.csv
  ✓ rangers_pitcher_gamelogs.csv
  ✓ rangers_roster_2025.csv
  ✓ rangers_running.csv

✅  전처리 완료
